In [ ]:
import os
import cv2
import numpy as np
import torch
from torch import nn
from torchvision.transforms import v2
import torch.nn.functional as F
from PIL import Image

if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

transform = v2.Compose([
    v2.Resize((128, 128)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True)
]) 

class CNN(nn.Module):
    def __init__(self, pool_size=(2, 2), num_classes=1, dropout_p=0.5):
        super().__init__()

        # declaring the layers (3 channels (rgb), output 32 channels)
        self.conv1 = nn.Conv2d(3, 32, kernel_size = 3, padding = 1)
        self.bn1 = nn.BatchNorm2d(num_features=32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size = 3, padding = 1)
        self.bn2 = nn.BatchNorm2d(num_features=64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size = 3, padding = 1) 
        self.bn3 = nn.BatchNorm2d(num_features=128)

        self.pooling = nn.MaxPool2d(kernel_size=pool_size)
        self.dropout = nn.Dropout(p=dropout_p)
        self.flatten = nn.Flatten()
        # self.linear = nn.Linear((128 * 14 * 14), 128) # without padding
        self.linear = nn.Linear((128 * 16 * 16), 128) # with padding
        self.output = nn.Linear(128, num_classes)

    def forward(self, x):
        '''
        Foward (conv -> bn -> relu -> pool -> dropout)
        '''
        x = self.conv1(x) # (3, 128, 128) -> (32, 126, 126) (without padding
        # pooling reduce the size, but keeps the features
        x = self.bn1(x)
        x = F.relu(x) # use nn.functional is better ... for some reason
        # conv increase # of features, but does not change the size
        x = self.pooling(x) # -> (32, 63, 63) 
        x = self.dropout(x)

        x = self.conv2(x) # -> (64, 61, 61) (without padding
        x = self.bn2(x)
        x = F.relu(x) # activation function
        x = self.pooling(x) # -> (64, 30, 30)
        x = self.dropout(x)

        x = self.conv3(x) # -> (128, 28, 28) (without padding
        x = self.bn3(x)
        x = F.relu(x) # activation function
        x = self.pooling(x) # -> (128, 14, 14) -> final conv result 
        x = self.dropout(x)

        x = self.flatten(x)
        x = self.linear(x)
        x = F.relu(x)
        x = self.output(x)

        # use BCEWithLogitsLoss instead of sigmoid function
        # x = self.sigmoid(x)

        return x

model = CNN()
state_dict = torch.load("model.pth", weights_only=True)
model.load_state_dict(state_dict)
model.eval()
model = model.to(device)

import matplotlib.pyplot as plt

def predict_image(image_path):
    image = Image.open(image_path).convert("RGB")
    plt.imshow(image)
    plt.axis("off")
    plt.show()
    image = transform(image).to(device)
    output = model(image.unsqueeze(0))
    if output >= 0.5:
        prediction = "cat"
    else:
        prediction = "dog"
    print("The model predicts: " + prediction)

In [ ]:
predict_image("imgs/cat1.jpg")